# Notebook 10 — novas famílias de base: BalancedRF + LGBM-DART (quarentena)

TabPFN foi descartado (limite 10 classes + token). Rota alternativa: duas famílias com viés indutivo diferente das 4 do meta E12, ambas 16 classes nativas, CPU baratas, conda-only:
(a) `BalancedRandomForest` (imbalanced-learn: balanceamento por árvore > `class_weight` global para recall das raras);
(b) LightGBM com `boosting_type='dart'` (dropout em árvores, dinâmica diferente do GBDT/LGBM-GBDT).
OOF probas (16) de cada uma entram no meta como 5ª/6ª famílias. Adotar só com delta 5/5. Modo rápido: `NB10_MODO=rapido` (2 folds).

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
K = 16
TARGET = 'AGE_RANGE'

def _data_dir():
    for cand in [os.path.join('..', '..', 'data'), os.path.join('..', 'data'), 'data']:
        if os.path.isdir(cand):
            return cand
    raise FileNotFoundError('pasta data/ nao encontrada')
DATA_DIR = _data_dir()

MODO = os.environ.get('NB10_MODO', 'completo')
SUF = '_rap' if MODO in ('rapido', 'ecra') else ''
N_SPLITS = 2 if SUF else 5

train = pd.read_csv(os.path.join(DATA_DIR, 'train_nb02.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test_nb02.csv'))
y = train[TARGET].to_numpy().astype(int)
XBASE = train.drop(columns=['PID', TARGET])
Xte = test.drop(columns=['PID'])
assert list(Xte.columns) == list(XBASE.columns), 'teste desalinhado'
Xv = XBASE.to_numpy(dtype=np.float64)
Xtev = Xte.to_numpy(dtype=np.float64)
FOLDS = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(XBASE, y))
print('MODO=', MODO, '| X:', XBASE.shape, '| teste:', Xte.shape)

RESULTADOS = []
def registar(exp, variante, f1s, base_f1s, nota=''):
    f1s = np.asarray(f1s, float); base_f1s = np.asarray(base_f1s, float)
    r = {'exp': exp, 'variante': variante, 'f1': float(f1s.mean()), 'std': float(f1s.std()),
         'delta': float(f1s.mean() - base_f1s.mean()), 'folds_ganhos': int((f1s > base_f1s).sum()), 'nota': nota}
    RESULTADOS.append(r)
    print(r)
    return r

def oof_loop(make, Xtr, ytr, Xva_list, tag):
    p = os.path.join(DATA_DIR, f'nb10_{tag}{SUF}.npz')
    if os.path.exists(p):
        d = np.load(p)
        print(f'{tag}: cache {p}')
        return d['f1s'], d['oof'], d['proba']
    t0 = time.time()
    f1s, oof, proba = [], np.zeros(len(y), dtype=int), np.zeros((len(y), K))
    for fi, (tr, va) in enumerate(FOLDS):
        t1 = time.time()
        m = make()
        m.fit(Xtr[tr], ytr[tr])
        pr = m.predict(Xtr[va])
        oof[va] = pr
        proba[va] = m.predict_proba(Xtr[va])
        f1s.append(f1_score(ytr[va], pr, average='macro'))
        print(f'  fold {fi}: {f1s[-1]:.4f} ({time.time()-t1:.0f}s / total {time.time()-t0:.0f}s)', flush=True)
    f1s = np.array(f1s)
    np.savez_compressed(p, f1s=f1s, oof=oof, proba=proba)
    return f1s, oof, proba

def test_proba(make, tag):
    p = os.path.join(DATA_DIR, f'nb10_{tag}_te{SUF}.npz')
    if os.path.exists(p):
        return np.load(p)['proba_te']
    acc = np.zeros((len(Xtev), K))
    for tr, va in FOLDS:
        m = make()
        m.fit(Xv[tr], y[tr])
        acc += m.predict_proba(Xtev) / len(FOLDS)
    np.savez_compressed(p, proba_te=acc)
    return acc

## (a) BalancedRandomForest — OOF + teste

In [ ]:
from imblearn.ensemble import BalancedRandomForestClassifier

def make_brf():
    return BalancedRandomForestClassifier(n_estimators=500, sampling_strategy='auto',
        replacement=True, random_state=RANDOM_STATE, n_jobs=-1)
f1s_brf, oof_brf, proba_brf = oof_loop(make_brf, Xv, y, None, 'brf')
proba_brf_te = test_proba(make_brf, 'brf')
print(f'BRF standalone: {f1s_brf.mean():.4f} +/- {f1s_brf.std():.4f}')

## (b) LightGBM-DART — OOF + teste

Mesmo esqueleto do E16 (500, leaves 31, lr 0.05, feature_fraction 0.8) mas `boosting_type='dart'` (sem bagging: incompatível com DART).

In [ ]:
import lightgbm as lgb

def make_dart():
    return lgb.LGBMClassifier(n_estimators=500, num_leaves=31, learning_rate=0.05,
        feature_fraction=0.8, bagging_fraction=1.0, bagging_freq=0,
        boosting_type='dart', drop_rate=0.1, skip_drop=0.5,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
f1s_dart, oof_dart, proba_dart = oof_loop(make_dart, Xv, y, None, 'dart')
proba_dart_te = test_proba(make_dart, 'dart')
print(f'DART standalone: {f1s_dart.mean():.4f} +/- {f1s_dart.std():.4f}')

## Meta aumentado (64+16+16=96)

Mesmo protocolo E12 (logreg C=0.1, balanced). Referência: média C=0.1 de `nb08_e12_metastack.json`.

In [ ]:
zx = np.load(os.path.join(DATA_DIR, 'xgb_oof.npz'))['proba']
zh = np.load(os.path.join(DATA_DIR, 'combined_oof.npz'))['proba']
zf = np.load(os.path.join(DATA_DIR, 'nb08_e1_proba.npz'))['proba']
ze = np.load(os.path.join(DATA_DIR, 'nb08_e4_proba.npz'))['proba']
M96 = np.hstack([zx, zh, zf, ze, proba_brf, proba_dart])
print('meta-features:', M96.shape)
try:
    ref = float(np.mean(json.load(open(os.path.join(DATA_DIR, 'nb08_e12_metastack.json')))['logreg C=0.1']))
    print('referencia E12 C=0.1:', round(ref, 4))
except Exception as e:
    print('sem referencia E12:', e)
f1s_meta = []
for tr, va in FOLDS:
    meta = LogisticRegression(C=0.1, class_weight='balanced', max_iter=2000)
    meta.fit(M96[tr], y[tr])
    f1s_meta.append(f1_score(y[va], meta.predict(M96[va]), average='macro'))
f1s_meta = np.array(f1s_meta)
print(f'meta96: {f1s_meta.mean():.4f} +/- {f1s_meta.std():.4f}')
pd.DataFrame([{'exp': 'meta96', 'variante': 'E12 + BRF + DART',
                'f1': round(float(f1s_meta.mean()), 4), 'std': round(float(f1s_meta.std()), 4)}]).to_csv(
    os.path.join(DATA_DIR, f'nb10_resultados{SUF}.csv'), index=False)